# NOEMA · continuar la auto-mejora
Sube el ZIP actualizado. Una celda instala, entrena, evalúa y descarga resultados. La ejecución medida incluida es CPU; Colab no se ejecutó desde este entorno.

In [1]:
# Ejecuta esta única celda y selecciona AGI-Code-AGI.zip.
GENERATIONS = 100
DEVICE = "cpu"  # Puedes usar "cuda" en una sesión con GPU.
CONTINUE_TRAINED = True

from google.colab import files
from pathlib import Path
import subprocess, sys, zipfile, tempfile, shutil

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if zip_name is None:
    raise ValueError("Selecciona el ZIP del proyecto")
work = Path(tempfile.mkdtemp(prefix="noema_"))
with zipfile.ZipFile(zip_name) as archive:
    for item in archive.infolist():
        target = (work / item.filename).resolve()
        if not target.is_relative_to(work.resolve()):
            raise ValueError("Ruta no válida en el ZIP")
        if (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Enlace no permitido en el ZIP")
    archive.extractall(work)
project = next(work.rglob("run_rsi.py")).parent
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(project)])
output = work / "resultados"
command = [sys.executable, str(project / "run_rsi.py"), "--generations", str(GENERATIONS),
           "--device", DEVICE, "--output", str(output)]
checkpoint = project / "examples/trained_run/checkpoint.pt"
if CONTINUE_TRAINED and checkpoint.exists():
    command += ["--resume", str(checkpoint)]
subprocess.check_call(command, cwd=project)
print((output / "report.json").read_text()[:2200])
archive_path = shutil.make_archive(str(work / "NOEMA_resultados"), "zip", output)
files.download(archive_path)


Saving ASI.zip to ASI.zip
{
  "format_version": 1,
  "scope": "Grounded next-state prediction in PhysicsPlayground",
  "generation": 10,
  "promotions": 4,
  "config": {
    "seed": 42,
    "generations": 100,
    "candidates": 3,
    "train_steps": 150,
    "batch_size": 64,
    "initial_episodes": 12,
    "fresh_episodes": 6,
    "eval_episodes": 20,
    "horizon": 24,
    "replay_capacity": 12000,
    "auxiliary_every": 8,
    "min_improvement": 0.005,
    "max_task_regression": 0.02,
    "patience": 3,
    "threads": 1,
    "device": "cpu"
  },
  "initial_validation": {
    "mse": 0.4347419738769531,
    "raw_mse": 0.03674134239554405,
    "persistence_mse": 0.4347418546676636,
    "per_task": {
      "free_fall": 0.18944759666919708,
      "impacts": 0.6387518644332886,
      "control": 0.47602638602256775
    },
    "episode_errors": [
      0.2654961049556732,
      0.20022796094417572,
      0.1657559722661972,
      0.12249886244535446,
      0.20131796598434448,
      0.42916

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>